In [ ]:
%pip install matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
jobs = pd.read_csv("../data/processed/jobs_final.csv")

applications = pd.read_csv(
    "../data/processed/applications.csv"
)

application_analysis = pd.read_csv(
    "../data/processed/application_analysis.csv"
)

print("Jobs:", jobs.shape)
print("Applications:", applications.shape)
print("Application analysis:", application_analysis.shape)

In [ ]:
print("JOBS COLUMNS")
print(jobs.columns.tolist())

print("\nAPPLICATIONS COLUMNS")
print(applications.columns.tolist())

print("\nAPPLICATION ANALYSIS COLUMNS")
print(application_analysis.columns.tolist())

In [ ]:
print("APPLICATION ANALYSIS INFO")
print("-" * 50)

application_analysis.info()

print("\nMISSING VALUES")
print("-" * 50)

missing_summary = pd.DataFrame({
    "missing_count": application_analysis.isna().sum(),
    "missing_pct": (
        application_analysis.isna().mean() * 100
    ).round(2)
})

print(missing_summary)

In [ ]:
application_analysis["application_date"] = pd.to_datetime(
    application_analysis["application_date"],
    errors="coerce"
)

application_analysis["first_response_date"] = pd.to_datetime(
    application_analysis["first_response_date"],
    errors="coerce"
)

print(application_analysis[
    ["application_date", "first_response_date"]
].dtypes)

print("\nApplication date range:")
print("Earliest:", application_analysis["application_date"].min())
print("Latest:", application_analysis["application_date"].max())

print("\nFirst response date range:")
print("Earliest:", application_analysis["first_response_date"].min())
print("Latest:", application_analysis["first_response_date"].max())

In [ ]:
print("APPLICATION PORTFOLIO OVERVIEW")
print("-" * 50)

print("Total applications:", len(application_analysis))

print(
    "Unique jobs:",
    application_analysis["job_id"].nunique()
)

print(
    "Unique companies:",
    application_analysis["company_name"].nunique()
)

print(
    "Applications with known date:",
    application_analysis["application_date"].notna().sum()
)

print(
    "Applications missing date:",
    application_analysis["application_date"].isna().sum()
)

print("\nSTATUS DISTRIBUTION")
print("-" * 50)
print(
    application_analysis["status"]
    .value_counts(dropna=False)
)

print("\nRESPONSE STATUS DISTRIBUTION")
print("-" * 50)
print(
    application_analysis["response_status"]
    .value_counts(dropna=False)
)

print("\nROLE CATEGORY DISTRIBUTION")
print("-" * 50)
print(
    application_analysis["role_category"]
    .value_counts(dropna=False)
)

print("\nCAREER LEVEL DISTRIBUTION")
print("-" * 50)
print(
    application_analysis["career_level"]
    .value_counts(dropna=False)
)

In [ ]:
total_applications = len(application_analysis)
unique_jobs = application_analysis["job_id"].nunique()
unique_companies = application_analysis["company_name"].nunique()

responded = (
    application_analysis["response_status"] == "Responded"
).sum()

no_response = (
    application_analysis["response_status"] == "No Response"
).sum()

rejected = (
    application_analysis["status"] == "Rejected"
).sum()

response_rate = responded / total_applications * 100
rejection_rate = rejected / total_applications * 100

applications_per_company = (
    total_applications / unique_companies
)

print("CORE APPLICATION KPIs")
print("-" * 50)

print(f"Total applications: {total_applications}")
print(f"Unique jobs: {unique_jobs}")
print(f"Unique companies: {unique_companies}")

print(f"\nObserved responses: {responded}")
print(f"No observed response: {no_response}")
print(f"Observed rejections: {rejected}")

print(f"\nObserved response rate: {response_rate:.2f}%")
print(f"Observed rejection rate: {rejection_rate:.2f}%")

print(
    f"Average applications per company: "
    f"{applications_per_company:.2f}"
)

In [ ]:
monthly_applications = (
    application_analysis
    .dropna(subset=["application_date"])
    .assign(
        application_month=lambda x:
            x["application_date"].dt.to_period("M")
    )
    .groupby("application_month")
    .size()
    .reset_index(name="applications")
)

monthly_applications["application_month"] = (
    monthly_applications["application_month"].dt.to_timestamp()
)

print(monthly_applications)

print(
    "\nTotal applications represented:",
    monthly_applications["applications"].sum()
)

In [ ]:
full_month_range = pd.date_range(
    start=monthly_applications["application_month"].min(),
    end=monthly_applications["application_month"].max(),
    freq="MS"
)

monthly_applications = (
    monthly_applications
    .set_index("application_month")
    .reindex(full_month_range, fill_value=0)
    .rename_axis("application_month")
    .reset_index()
)

print(monthly_applications)

print(
    "\nTotal applications represented:",
    monthly_applications["applications"].sum()
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    monthly_applications["application_month"],
    monthly_applications["applications"],
    marker="o",
    linewidth=2
)

ax.set_title(
    "Monthly Job Application Activity",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Application Month")
ax.set_ylabel("Number of Applications")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax.tick_params(
    axis="x",
    rotation=45
)

for x, y in zip(
    monthly_applications["application_month"],
    monthly_applications["applications"]
):
    ax.annotate(
        str(y),
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=9
    )

fig.tight_layout()

plt.savefig(
    "../reports/figures/monthly_application_activity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
role_distribution = (
    application_analysis["role_category"]
    .value_counts()
    .rename_axis("role_category")
    .reset_index(name="applications")
)

role_distribution["percentage"] = (
    role_distribution["applications"]
    / total_applications
    * 100
).round(2)

print(role_distribution)
print(
    "\nTotal applications:",
    role_distribution["applications"].sum()
)

In [ ]:
role_plot = role_distribution.sort_values(
    "applications",
    ascending=True
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    role_plot["role_category"],
    role_plot["applications"]
)

ax.set_title(
    "Applications by Role Category",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Number of Applications")
ax.set_ylabel("Role Category")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, (count, pct) in enumerate(
    zip(
        role_plot["applications"],
        role_plot["percentage"]
    )
):
    ax.text(
        count + 2,
        i,
        f"{count} ({pct:.1f}%)",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    role_plot["applications"].max() * 1.18
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/applications_by_role_category.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
career_level_distribution = (
    application_analysis["career_level"]
    .value_counts()
    .rename_axis("career_level")
    .reset_index(name="applications")
)

career_level_distribution["percentage"] = (
    career_level_distribution["applications"]
    / total_applications
    * 100
).round(2)

print(career_level_distribution)

print(
    "\nTotal applications:",
    career_level_distribution["applications"].sum()
)

In [ ]:
career_plot = career_level_distribution.sort_values(
    "applications",
    ascending=True
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    career_plot["career_level"],
    career_plot["applications"]
)

ax.set_title(
    "Applications by Career Level",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Number of Applications")
ax.set_ylabel("Career Level")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, (count, pct) in enumerate(
    zip(
        career_plot["applications"],
        career_plot["percentage"]
    )
):
    ax.text(
        count + 2,
        i,
        f"{count} ({pct:.1f}%)",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    career_plot["applications"].max() * 1.18
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/applications_by_career_level.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
role_outcomes = (
    application_analysis
    .groupby("role_category")
    .agg(
        applications=("application_id", "count"),
        observed_rejections=(
            "status",
            lambda x: (x == "Rejected").sum()
        )
    )
    .reset_index()
)

role_outcomes["observed_rejection_rate"] = (
    role_outcomes["observed_rejections"]
    / role_outcomes["applications"]
    * 100
).round(2)

role_outcomes = role_outcomes.sort_values(
    "applications",
    ascending=False
)

print(role_outcomes)

In [ ]:
rejection_plot = (
    role_outcomes[
        role_outcomes["role_category"] != "Unknown"
    ]
    .sort_values(
        "observed_rejections",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    rejection_plot["role_category"],
    rejection_plot["observed_rejections"]
)

ax.set_title(
    "Observed Rejections by Role Category",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Observed Rejections")
ax.set_ylabel("Role Category")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, (count, rate) in enumerate(
    zip(
        rejection_plot["observed_rejections"],
        rejection_plot["observed_rejection_rate"]
    )
):
    ax.text(
        count + 0.3,
        i,
        f"{count} ({rate:.1f}%)",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    rejection_plot["observed_rejections"].max() * 1.25
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/observed_rejections_by_role_category.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
timing_data = application_analysis[
    (application_analysis["status"] == "Rejected")
    & application_analysis["application_date"].notna()
    & application_analysis["first_response_date"].notna()
].copy()

timing_data["days_to_response"] = (
    timing_data["first_response_date"]
    - timing_data["application_date"]
).dt.days

print("Rejected applications with measurable timing:", len(timing_data))

print("\nDays to response:")
print(
    timing_data["days_to_response"]
    .describe()
)

print("\nIndividual observations:")
print(
    timing_data[
        [
            "application_id",
            "application_date",
            "first_response_date",
            "days_to_response"
        ]
    ].sort_values("days_to_response")
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    timing_data["days_to_response"],
    bins=range(
        timing_data["days_to_response"].min(),
        timing_data["days_to_response"].max() + 3,
        3
    ),
    edgecolor="black"
)

ax.axvline(
    timing_data["days_to_response"].median(),
    linestyle="--",
    linewidth=2,
    label=f"Median: {timing_data['days_to_response'].median():.1f} days"
)

ax.axvline(
    timing_data["days_to_response"].mean(),
    linestyle=":",
    linewidth=2,
    label=f"Mean: {timing_data['days_to_response'].mean():.1f} days"
)

ax.set_title(
    "Time to Observed Rejection",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Days from Application to First Response")
ax.set_ylabel("Number of Applications")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax.legend()

fig.tight_layout()

plt.savefig(
    "../reports/figures/days_to_observed_rejection.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
company_distribution = (
    application_analysis
    .groupby("company_name")
    .size()
    .reset_index(name="applications")
    .sort_values(
        "applications",
        ascending=False
    )
)

print("Companies:", len(company_distribution))

print("\nTop 20 companies by application count:")
print(
    company_distribution.head(20).to_string(index=False)
)

print("\nApplication count distribution per company:")
print(
    company_distribution["applications"].value_counts()
    .sort_index()
)

In [ ]:
top_companies = (
    company_distribution
    .head(15)
    .sort_values("applications", ascending=True)
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    top_companies["company_name"],
    top_companies["applications"]
)

ax.set_title(
    "Top Companies by Application Volume",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Number of Applications")
ax.set_ylabel("Company")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, count in enumerate(top_companies["applications"]):
    ax.text(
        count + 0.3,
        i,
        str(count),
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    top_companies["applications"].max() * 1.18
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/top_companies_by_application_volume.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
single_application_companies = (
    company_distribution["applications"] == 1
).sum()

multi_application_companies = (
    company_distribution["applications"] >= 2
).sum()

applications_to_multi_companies = (
    company_distribution.loc[
        company_distribution["applications"] >= 2,
        "applications"
    ].sum()
)

top_5_applications = (
    company_distribution
    .head(5)["applications"]
    .sum()
)

single_company_pct = (
    single_application_companies
    / len(company_distribution)
    * 100
)

multi_company_pct = (
    multi_application_companies
    / len(company_distribution)
    * 100
)

multi_application_pct = (
    applications_to_multi_companies
    / total_applications
    * 100
)

top_5_application_pct = (
    top_5_applications
    / total_applications
    * 100
)

print("COMPANY CONCENTRATION")
print("-" * 50)

print(
    f"Companies with exactly 1 application: "
    f"{single_application_companies}"
)

print(
    f"Companies with 2+ applications: "
    f"{multi_application_companies}"
)

print(
    f"Percentage of companies with exactly 1 application: "
    f"{single_company_pct:.2f}%"
)

print(
    f"Percentage of companies with 2+ applications: "
    f"{multi_company_pct:.2f}%"
)

print(
    f"\nApplications sent to companies with 2+ applications: "
    f"{applications_to_multi_companies}"
)

print(
    f"Percentage of applications sent to repeat-targeted companies: "
    f"{multi_application_pct:.2f}%"
)

print(
    f"\nTop 5 companies' applications: "
    f"{top_5_applications}"
)

print(
    f"Top 5 companies' share of all applications: "
    f"{top_5_application_pct:.2f}%"
)

In [ ]:
role_outcomes_reliable = (
    role_outcomes[
        role_outcomes["applications"] >= 10
    ]
    .copy()
    .sort_values(
        "observed_rejection_rate",
        ascending=False
    )
)

print(
    role_outcomes_reliable.to_string(index=False)
)

In [ ]:
rejection_rate_plot = (
    role_outcomes_reliable[
        role_outcomes_reliable["role_category"] != "Unknown"
    ]
    .sort_values(
        "observed_rejection_rate",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    rejection_rate_plot["role_category"],
    rejection_rate_plot["observed_rejection_rate"]
)

ax.set_title(
    "Observed Rejection Rate by Role Category",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Observed Rejection Rate (%)")
ax.set_ylabel("Role Category")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, (rate, count) in enumerate(
    zip(
        rejection_rate_plot["observed_rejection_rate"],
        rejection_rate_plot["applications"]
    )
):
    ax.text(
        rate + 0.5,
        i,
        f"{rate:.1f}% (n={count})",
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    rejection_rate_plot["observed_rejection_rate"].max() * 1.2
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/observed_rejection_rate_by_role.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
monthly_role = (
    application_analysis
    .dropna(subset=["application_date"])
    .assign(
        application_month=lambda x:
            x["application_date"].dt.to_period("M")
            .dt.to_timestamp()
    )
    .groupby(
        ["application_month", "role_category"]
    )
    .size()
    .reset_index(name="applications")
)

monthly_role_pivot = (
    monthly_role
    .pivot(
        index="application_month",
        columns="role_category",
        values="applications"
    )
    .fillna(0)
)

print(monthly_role_pivot)

In [ ]:
major_role_categories = [
    "Unknown",
    "Software Engineering",
    "Data Engineering",
    "Data / Analytics Analyst",
    "Data Science"
]

fig, ax = plt.subplots(figsize=(12, 6))

for role in major_role_categories:
    ax.plot(
        monthly_role_pivot.index,
        monthly_role_pivot[role],
        marker="o",
        linewidth=2,
        label=role
    )

ax.set_title(
    "Role Targeting Over Time",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Application Month")
ax.set_ylabel("Number of Applications")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax.tick_params(
    axis="x",
    rotation=45
)

ax.legend(
    title="Role Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/role_targeting_over_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
monthly_role_pct = monthly_role_pivot.div(
    monthly_role_pivot.sum(axis=1),
    axis=0
) * 100

print(
    monthly_role_pct.round(1).to_string()
)

In [ ]:
role_mix_comparison = (
    monthly_role_pct
    .loc[
        [
            pd.Timestamp("2026-08-01"),
            pd.Timestamp("2026-09-01")
        ],
        major_role_categories
    ]
)

# Combine all smaller categories into "Other"
role_mix_comparison["Other"] = (
    100 - role_mix_comparison[major_role_categories].sum(axis=1)
)

print(
    role_mix_comparison.round(1)
)

In [ ]:
role_mix_plot = role_mix_comparison.copy()

# Rename index for cleaner chart labels
role_mix_plot.index = ["August 2026", "September 2026"]

fig, ax = plt.subplots(figsize=(11, 6))

bottom = np.zeros(len(role_mix_plot))

plot_categories = [
    "Unknown",
    "Software Engineering",
    "Data Engineering",
    "Data / Analytics Analyst",
    "Data Science",
    "Other"
]

for role in plot_categories:
    values = role_mix_plot[role].values

    ax.bar(
        role_mix_plot.index,
        values,
        bottom=bottom,
        label=role
    )

    bottom += values

ax.set_title(
    "Application Role Mix: August vs. September 2026",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Application Month")
ax.set_ylabel("Share of Applications (%)")

ax.set_ylim(0, 100)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

ax.legend(
    title="Role Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/role_mix_august_vs_september.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
title_quality = (
    application_analysis
    .groupby("role_category")
    .agg(
        applications=("application_id", "count"),
        titles_present=("job_title", "count"),
        cleaned_titles_present=("job_title_clean", "count")
    )
    .reset_index()
)

title_quality["title_coverage_pct"] = (
    title_quality["titles_present"]
    / title_quality["applications"]
    * 100
).round(2)

title_quality["cleaned_title_coverage_pct"] = (
    title_quality["cleaned_titles_present"]
    / title_quality["applications"]
    * 100
).round(2)

title_quality = title_quality.sort_values(
    "applications",
    ascending=False
)

print(title_quality.to_string(index=False))

In [ ]:
title_distribution = (
    application_analysis
    .dropna(subset=["job_title_clean"])
    .groupby("job_title_clean")
    .size()
    .reset_index(name="applications")
    .sort_values(
        "applications",
        ascending=False
    )
)

print(
    "Unique cleaned job titles:",
    title_distribution["job_title_clean"].nunique()
)

print("\nTop 20 cleaned job titles:")
print(
    title_distribution.head(20).to_string(index=False)
)

In [ ]:
top_titles = (
    title_distribution
    .head(15)
    .sort_values(
        "applications",
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(
    top_titles["job_title_clean"],
    top_titles["applications"]
)

ax.set_title(
    "Most Frequently Targeted Job Titles",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Number of Applications")
ax.set_ylabel("Cleaned Job Title")

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, count in enumerate(top_titles["applications"]):
    ax.text(
        count + 0.15,
        i,
        str(count),
        va="center",
        fontsize=9
    )

ax.set_xlim(
    0,
    top_titles["applications"].max() * 1.25
)

fig.tight_layout()

plt.savefig(
    "../reports/figures/top_targeted_job_titles.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
email_summary = (
    application_analysis
    .groupby("response_status")
    .agg(
        applications=("application_id", "count"),
        total_emails=("email_count", "sum"),
        average_emails=("email_count", "mean"),
        median_emails=("email_count", "median"),
        max_emails=("email_count", "max")
    )
    .reset_index()
)

print(email_summary)

print("\nOverall email count distribution:")
print(
    application_analysis["email_count"].describe()
)

In [ ]:
eda_kpis = {
    "total_applications": total_applications,
    "unique_jobs": unique_jobs,
    "unique_companies": unique_companies,

    "applications_with_known_date": int(
        application_analysis["application_date"].notna().sum()
    ),

    "applications_without_known_date": int(
        application_analysis["application_date"].isna().sum()
    ),

    "observed_rejections": int(rejected),

    "observed_response_rate_pct": round(
        response_rate, 2
    ),

    "observed_rejection_rate_pct": round(
        rejection_rate, 2
    ),

    "companies_with_one_application": int(
        single_application_companies
    ),

    "companies_with_multiple_applications": int(
        multi_application_companies
    ),

    "top_5_company_application_share_pct": round(
        top_5_application_pct, 2
    ),

    "applications_aug_sep_2026": int(
        monthly_applications[
            monthly_applications["application_month"].isin(
                [
                    pd.Timestamp("2026-08-01"),
                    pd.Timestamp("2026-09-01")
                ]
            )
        ]["applications"].sum()
    ),

    "measurable_rejection_timings": len(timing_data),

    "median_days_to_response": float(
        timing_data["days_to_response"].median()
    ),

    "mean_days_to_response": float(
        timing_data["days_to_response"].mean()
    ),

    "unique_cleaned_job_titles": int(
        title_distribution["job_title_clean"].nunique()
    )
}

eda_kpi_table = (
    pd.Series(eda_kpis)
    .rename("value")
    .to_frame()
)

print(eda_kpi_table)

In [ ]:
print("FINAL EDA QUALITY CHECK")
print("-" * 50)

print(
    "Duplicate application IDs:",
    application_analysis["application_id"].duplicated().sum()
)

print(
    "Duplicate job IDs:",
    application_analysis["job_id"].duplicated().sum()
)

print(
    "Negative email counts:",
    (application_analysis["email_count"] < 0).sum()
)

print(
    "Responses before applications:",
    (
        (
            application_analysis["first_response_date"]
            < application_analysis["application_date"]
        )
        .fillna(False)
        .sum()
    )
)

print(
    "Invalid days-to-response values:",
    (
        timing_data["days_to_response"] < 0
    ).sum()
)

print(
    "\nFigure files expected: 10"
)